<a href="https://colab.research.google.com/github/SaharMian/recsys-letterboxd/blob/main/notebooks/twotower.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie recommendations on Letterboxd data, part 3: a two-tower model with film text

The previous models only knew films as ids. This notebook adds **content**: a text description of each film (title, year, genres) is turned into a vector with a pretrained language model, and a **two-tower** model uses it. This should help with films that have few or no ratings (the *cold start* problem), which we test separately at the end.

**This notebook covers:**
1. Loading the split and re-defining the evaluation
2. Matching our films to IMDb to get genres
3. Turning film text into vectors (sentence embeddings)
4. Building the two-tower model and training it
5. Adding the result to the comparison table
6. Cold start experiment: recommending films the model has never seen ratings for, with ablations
7. Final evaluation on the test set
8. Exporting recommendations for a small demo app

**Data needed in the `recsys` folder:** everything from parts 1 and 2, plus `title.basics.tsv` from [IMDb's datasets](https://datasets.imdbws.com/) (download `title.basics.tsv.gz`, unzip it and upload the `.tsv` file).

## Setup

Mount Google Drive, import the libraries and load the saved split from part 1. Use a GPU runtime (T4).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

BASE = '/content/drive/MyDrive/recsys/'
SEED, LIKE, K = 42, 4.0, 10
device = 'cuda'
print("GPU available:", torch.cuda.is_available())

data = pd.read_parquet(BASE + 'split_3000.parquet')
data['u'] = data['user_name'].cat.codes
data['i'] = data['film_id'].cat.codes
n_users, n_films = data['u'].nunique(), data['i'].nunique()

train = data[data['split'] == 'train']
val   = data[data['split'] == 'val']
test  = data[data['split'] == 'test']

# Films each user has already rated in train (never recommended)
seen_train = train.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

print(n_users, n_films, data['split'].value_counts().to_dict())

Mounted at /content/drive
GPU available: True
3000 61555 {'train': 3552662, 'val': 444086, 'test': 442580}


## Evaluation and popularity baseline

The same evaluation as in parts 1 and 2, copied here so that the notebook can run on its own. The popularity baseline must again give NDCG@10 0.1841.

In [2]:
def relevant_sets(df):
    liked = df[df['rating'] >= LIKE]
    return liked.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def evaluate(recommend_fn, eval_df, seen):
    relevant = relevant_sets(eval_df)
    ndcgs, recalls, recommended = [], [], set()
    for user, rel in relevant.items():
        recs = recommend_fn(user, seen.get(user, set()), K)
        recommended.update(recs)
        hits = [1 if f in rel else 0 for f in recs]
        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), K)))
        ndcgs.append(dcg / idcg)
        recalls.append(sum(hits) / len(rel))
    return {'NDCG@10': np.mean(ndcgs), 'Recall@10': np.mean(recalls),
            'Coverage': len(recommended) / data['film_id'].nunique()}

# Popularity baseline: films ranked by number of likes in train
pop_rank = train[train['rating'] >= LIKE]['film_id'].value_counts().index.tolist()

def recommend_pop(user, seen, k):
    out = []
    for f in pop_rank:
        if f not in seen:
            out.append(f)
            if len(out) == k:
                break
    return out

print(evaluate(recommend_pop, val, seen_train))

{'NDCG@10': np.float64(0.18405938695253699), 'Recall@10': np.float64(0.03855764705308532), 'Coverage': 0.001364633254812769}


## Matching films to IMDb to get genres

The Letterboxd file only has a title and a year for each film. To give the model something to work with we add **genres** from IMDb's public `title.basics.tsv`.

The two datasets have no common id, so we match on a key made of the **normalized title** (lower case, only letters and digits) and the **year**. For IMDb we only keep film-like types (movie, TV movie, short, video) and one row per key.

About **85%** of our films get genres. The unmatched ones are mostly titles that are written differently in the two sources (for example "Se7en" vs "Seven"). The rest simply get their title and year without genres, so nothing breaks.

In [3]:
films = pd.read_csv(BASE + 'films.csv', engine='python', on_bad_lines='skip')
print("title.basics.tsv exists on Drive:", os.path.exists(BASE + 'title.basics.tsv'))

def norm(s):
    """Lower case, keep only letters and digits"""
    return re.sub(r'[^a-z0-9]+', '', str(s).lower())

def make_key(title, year):
    return title.map(norm) + '_' + year.astype(int).astype(str)

# Our 61,555 films with title and year
sample_films = pd.DataFrame({'film_id': data['film_id'].cat.categories})
sample_films = sample_films.merge(films[['film_id', 'film_name', 'year']], on='film_id', how='left')
sample_films['key'] = make_key(sample_films['film_name'], sample_films['year'].fillna(0))

# IMDb: only film-like types, with title, year and genres
imdb = pd.read_csv(BASE + 'title.basics.tsv', sep='\t',
                   usecols=['titleType', 'primaryTitle', 'startYear', 'genres'],
                   dtype=str, na_values='\\N')
imdb = imdb[imdb['titleType'].isin(['movie', 'tvMovie', 'short', 'video'])].dropna(subset=['startYear'])
imdb['key'] = make_key(imdb['primaryTitle'], imdb['startYear'])
imdb = imdb.drop_duplicates('key')[['key', 'genres']]

meta = sample_films.merge(imdb, on='key', how='left')
print(f"Films: {len(meta):,}")
print(f"Matched with genres: {meta['genres'].notna().mean():.1%}")
print("\nExamples of matched films:")
print(meta[meta['genres'].notna()].sample(5, random_state=SEED)[['film_name', 'year', 'genres']])
print("\nExamples of unmatched films:")
print(meta[meta['genres'].isna()].sample(5, random_state=SEED)[['film_name', 'year']])

title.basics.tsv exists on Drive: True
Films: 61,555
Matched with genres: 85.1%

Examples of matched films:
                  film_name    year                       genres
43582         Love You More  2008.0                  Drama,Short
15717     Chasing Happiness  2019.0  Biography,Documentary,Music
15486                Jekyll  2007.0       Horror,Sci-Fi,Thriller
48611        Borley Rectory  2017.0       Animation,Drama,Horror
57749  Harold's Going Stiff  2011.0                Comedy,Horror

Examples of unmatched films:
                        film_name    year
25048                       Se7en  1995.0
58890                Jungleground  1995.0
57807  Iria: Zeiram the Animation  1994.0
36998             Travolta and Me  1994.0
31211              The Gunfighter  2014.0


## Turning film text into vectors

For each film we build a short text, for example `Spider-Man 2 (2004). Genres: Action, Adventure, Sci-Fi`, and let a pretrained language model (`all-MiniLM-L6-v2` from the sentence-transformers library) turn it into a vector of 384 numbers. Texts with similar meaning end up close to each other, and `normalize_embeddings=True` makes the dot product between two vectors equal their cosine similarity.

This is how the model can know something about a film that has **no ratings**: its text vector still exists. The vectors are saved to Drive so we do not need to recompute them.

Sanity check at the end: the nearest neighbours of *Spider-Man 2* should be other Spider-Man films, and they are (Spider-Man 3, Spider-Man, The Amazing Spider-Man 2).

In [4]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

def make_text(row):
    """Title (year). Genres: ..."""
    title = row['film_name'] if pd.notna(row['film_name']) else row['film_id'].replace('-', ' ')
    year = f" ({int(row['year'])})" if pd.notna(row['year']) else ""
    genres = f". Genres: {row['genres'].replace(',', ', ')}" if pd.notna(row['genres']) else ""
    return f"{title}{year}{genres}"

meta['text'] = meta.apply(make_text, axis=1)
print(meta['text'].head(3).tolist())

st_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
film_emb = st_model.encode(meta['text'].tolist(), batch_size=256,
                           show_progress_bar=True, normalize_embeddings=True)
print(film_emb.shape)

np.save(BASE + 'film_text_emb.npy', film_emb)

# Sanity check: which films are closest to Spider-Man 2 in vector space?
q = meta.index[meta['film_id'] == 'spider-man-2']
q = q[0] if len(q) else 0
sims = film_emb @ film_emb[q]
top = np.argsort(-sims)[:6]
print("\nNearest neighbours of:", meta.loc[q, 'text'])
print(meta.loc[top[1:], 'text'].tolist())

['008: Operation Exterminate (1965). Genres: Action, Drama', '0.0MHz (2019). Genres: Horror', '0.5 mm (2014). Genres: Drama']


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/241 [00:00<?, ?it/s]

(61555, 384)

Nearest neighbours of: Spider-Man 2 (2004). Genres: Action, Adventure, Sci-Fi
['Spider-Man 3 (2007). Genres: Action, Adventure, Sci-Fi', 'Spider-Man (2002). Genres: Action, Adventure, Sci-Fi', 'The Amazing Spider-Man 2 (2014). Genres: Action, Adventure, Sci-Fi', 'Spider (2007). Genres: Action, Drama, Short', 'The Amazing Spider-Man (2012). Genres: Action, Sci-Fi']


## Training data and the user profile

The training data is built exactly as for NCF: every liked train rating is a positive example and each gets 4 random negatives (films the user has not rated).

The new part is the **user profile**. For every user we take the **mean of the text vectors of the films they liked in train**. It describes the user's taste in the same space as the films, and it works for any user who has liked at least one film. Only train ratings are used, so nothing from validation or test leaks into the profile.

In [5]:
film_emb = np.load(BASE + 'film_text_emb.npy')
film_text = torch.tensor(film_emb, dtype=torch.float32, device=device)

pos = train[train['rating'] >= LIKE]
pos_u = torch.tensor(pos['u'].values, dtype=torch.long, device=device)
pos_i = torch.tensor(pos['i'].values, dtype=torch.long, device=device)

# Which (user, film) pairs have a rating in train (never drawn as negatives)
seen_mask = torch.zeros(n_users, n_films, dtype=torch.bool, device=device)
seen_mask[torch.tensor(train['u'].values, dtype=torch.long, device=device),
          torch.tensor(train['i'].values, dtype=torch.long, device=device)] = True

def sample_negatives(u, n_neg=4):
    """For each user id in u, draw n_neg random films the user has not rated in train"""
    u = u.repeat_interleave(n_neg)
    neg = torch.randint(0, n_films, u.shape, device=device)
    bad = seen_mask[u, neg]
    while bad.any():
        neg[bad] = torch.randint(0, n_films, (int(bad.sum()),), device=device)
        bad = seen_mask[u, neg]
    return u, neg

# User profile: mean text vector of the films the user liked in train
user_profile = torch.zeros(n_users, film_text.shape[1], device=device)
user_profile.index_add_(0, pos_u, film_text[pos_i])
likes_per_user = torch.bincount(pos_u, minlength=n_users)
user_profile = user_profile / likes_per_user.clamp(min=1).unsqueeze(1)

print("film_text:", tuple(film_text.shape), "| user_profile:", tuple(user_profile.shape))
print("Positive examples:", f"{len(pos_u):,}", "| users without liked films in train:", int((likes_per_user == 0).sum()))

film_text: (61555, 384) | user_profile: (3000, 384)
Positive examples: 1,333,735 | users without liked films in train: 0


## The two-tower model and training

A two-tower model has one network for users and one for films. Each tower turns its input into a vector of 64 numbers, and the score for a (user, film) pair is the **dot product** of the two vectors.

- **User tower:** the user profile (384 numbers, taste described with text) plus a learned user id embedding.
- **Film tower:** the film's text vector (384 numbers) plus a learned film id embedding.

Because the towers are separate, we can compute all film vectors once and score every film for a user with a single matrix product, which is how real recommender systems retrieve candidates quickly.

**ID dropout:** during training the film id embedding is switched off (set to zero) for 30% of the examples. Without it the model would rely only on the id, which does not exist for a new film. With it, the model is forced to also learn to score films from their text alone.

Training is as for NCF: binary cross-entropy with 4 negatives per positive, and the best epoch is chosen on validation NDCG@10. In our run the best epoch was 10 (val NDCG 0.2249). The score was still rising at the last epoch, so a longer training might help a little; we kept 10 epochs to treat all neural models the same.

In [6]:
class TwoTower(nn.Module):
    def __init__(self, film_text, user_profile, d_id=32, d_out=64, p_id_drop=0.3):
        super().__init__()
        n_films, d_text = film_text.shape
        n_users = user_profile.shape[0]
        # fixed text vectors, not trained and not saved with the model
        self.register_buffer('film_text', film_text, persistent=False)
        self.register_buffer('user_profile', user_profile, persistent=False)
        self.user_id = nn.Embedding(n_users, d_id)
        self.film_id = nn.Embedding(n_films, d_id)
        nn.init.normal_(self.user_id.weight, std=0.01)
        nn.init.normal_(self.film_id.weight, std=0.01)
        self.user_mlp = nn.Sequential(nn.Linear(d_text + d_id, 128), nn.ReLU(), nn.Linear(128, d_out))
        self.film_mlp = nn.Sequential(nn.Linear(d_text + d_id, 128), nn.ReLU(), nn.Linear(128, d_out))
        self.p = p_id_drop

    def user_vec(self, u):
        return self.user_mlp(torch.cat([self.user_profile[u], self.user_id(u)], dim=1))

    def film_vec(self, i):
        fid = self.film_id(i)
        if self.training and self.p > 0:
            # ID dropout: switch off the film id for a share p of the examples
            fid = fid * (torch.rand(len(i), 1, device=i.device) > self.p).float()
        return self.film_mlp(torch.cat([self.film_text[i], fid], dim=1))

    def forward(self, u, i):
        return (self.user_vec(u) * self.film_vec(i)).sum(1)

def recommend_two_tower(model, exclude=('train',), k=K, chunk=500):
    """Top-k films per user, excluding the films the user has already seen"""
    ex = data[data['split'].isin(exclude)]
    ex_u = torch.tensor(ex['u'].values, dtype=torch.long, device=device)
    ex_i = torch.tensor(ex['i'].values, dtype=torch.long, device=device)
    users = data['user_name'].cat.categories
    films = data['film_id'].cat.categories
    model.eval()
    recs = {}
    with torch.no_grad():
        fv = model.film_vec(torch.arange(n_films, device=device))   # all film vectors, computed once
        for start in range(0, n_users, chunk):
            end = min(start + chunk, n_users)
            u = torch.arange(start, end, device=device)
            scores = model.user_vec(u) @ fv.T
            sel = (ex_u >= start) & (ex_u < end)
            scores[ex_u[sel] - start, ex_i[sel]] = float('-inf')    # already seen
            top = scores.topk(k, dim=1).indices.cpu().numpy()
            for row, uid in enumerate(range(start, end)):
                recs[users[uid]] = [films[j] for j in top[row]]
    return recs

N_NEG, BATCH, LR, EPOCHS = 4, 16384, 0.002, 10

torch.manual_seed(SEED)
tt = TwoTower(film_text, user_profile).to(device)
opt = torch.optim.Adam(tt.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()
best_ndcg, best_state, best_epoch = -1, None, 0

for epoch in range(1, EPOCHS + 1):
    tt.train()
    u_neg, i_neg = sample_negatives(pos_u, N_NEG)          # new negatives every epoch
    u_all = torch.cat([pos_u, u_neg])
    i_all = torch.cat([pos_i, i_neg])
    y_all = torch.cat([torch.ones(len(pos_u)), torch.zeros(len(u_neg))]).to(device)
    perm = torch.randperm(len(y_all), device=device)

    total = 0.0
    for s in range(0, len(perm), BATCH):
        idx = perm[s:s + BATCH]
        loss = loss_fn(tt(u_all[idx], i_all[idx]), y_all[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)

    recs_tt = recommend_two_tower(tt)
    res = evaluate(lambda user, seen, k: recs_tt[user], val, seen_train)
    print(f"Epoch {epoch:>2}: loss {total / len(perm):.4f} | val NDCG {res['NDCG@10']:.4f}, "
          f"Recall {res['Recall@10']:.4f}, Coverage {res['Coverage']:.4f}")
    if res['NDCG@10'] > best_ndcg:
        best_ndcg, best_state, best_epoch = res['NDCG@10'], copy.deepcopy(tt.state_dict()), epoch

tt.load_state_dict(best_state)
torch.save(best_state, BASE + 'two_tower_model.pt')
print(f"Best epoch: {best_epoch} (val NDCG {best_ndcg:.4f}), saved to Drive")

Epoch  1: loss 0.3181 | val NDCG 0.1687, Recall 0.0351, Coverage 0.0028
Epoch  2: loss 0.2635 | val NDCG 0.1925, Recall 0.0410, Coverage 0.0054
Epoch  3: loss 0.2416 | val NDCG 0.1900, Recall 0.0390, Coverage 0.0086
Epoch  4: loss 0.2294 | val NDCG 0.2026, Recall 0.0429, Coverage 0.0088
Epoch  5: loss 0.2202 | val NDCG 0.2062, Recall 0.0430, Coverage 0.0122
Epoch  6: loss 0.2126 | val NDCG 0.2168, Recall 0.0465, Coverage 0.0144
Epoch  7: loss 0.2057 | val NDCG 0.2116, Recall 0.0461, Coverage 0.0192
Epoch  8: loss 0.2004 | val NDCG 0.2149, Recall 0.0465, Coverage 0.0239
Epoch  9: loss 0.1952 | val NDCG 0.2234, Recall 0.0481, Coverage 0.0246
Epoch 10: loss 0.1915 | val NDCG 0.2249, Recall 0.0491, Coverage 0.0270
Best epoch: 10 (val NDCG 0.2249), saved to Drive


## Add the two-tower model to the comparison table

We add one row to `results_val.csv` (with a fixed name, so rerunning replaces it).

Result: the two-tower model reaches NDCG@10 **0.2249**. That is better than popularity (0.1841) and SVD (0.1929), but slightly below NCF (0.2303). On the normal task, where every film has plenty of ratings, the extra text information does not help beyond what the ids already tell the model. Its advantage should show for films **without** ratings, which is the next experiment.

In [7]:
recs_tt = recommend_two_tower(tt)
r = evaluate(lambda user, seen, k: recs_tt[user], val, seen_train)

tbl = pd.read_csv(BASE + 'results_val.csv', index_col=0)
tbl.loc['Two-tower'] = [round(r['NDCG@10'], 4), round(r['Recall@10'], 4), round(r['Coverage'], 4)]
tbl.to_csv(BASE + 'results_val.csv')
print(f"(best epoch: {best_epoch})")
tbl

(best epoch: 10)


,NDCG@10,Recall@10,Coverage
Popularity (likes),0.184059,0.038558,0.001365
SVD (min 0 ratings),0.008587,0.002159,0.032345
SVD (min 1000 ratings),0.175731,0.036463,0.003818
SVD (min 1500 ratings),0.192934,0.040607,0.000958
NCF (NeuMF),0.230300,0.052800,0.035400
Two-tower,0.224900,0.049100,0.027000


## Cold start experiment: 3,000 "new" films

In the normal split every test film also appears in train (we checked: 0% missing), so the test cannot show how a model handles a **brand new film**. We simulate it:

1. Pick 3,000 random films that have at least 20 train ratings (so they are ordinary films that users actually like).
2. **Remove all their ratings from train.** For the models, these films now have never been rated.
3. For each user, rank **only these 3,000 films** and check whether the ones the user liked in validation come out on top.

Because only 3,000 films are candidates, a random ranking is not zero but about 0.003. Three reference points:
- **Random:** the floor.
- **Content only (no training):** the user profile (mean text vector of liked films) compared directly to each film's text vector. No learning at all.
- **Two-tower:** trained on the reduced train set, then used on the new films.

The negatives during training must **never** include the new films (otherwise the model would learn that they are "not liked"). The user profile is also rebuilt from the reduced train set.

Note that the metric here is computed on the 2,853 users who liked at least one of the 3,000 films in validation, so the numbers are not comparable with the main table.

In [8]:
rng = np.random.default_rng(SEED)

# 3,000 "new" films with at least 20 ratings in train (ordinary films)
train_counts = train['i'].value_counts()
eligible = train_counts[train_counts >= 20].index.to_numpy()
cold_idx = rng.choice(eligible, size=3000, replace=False)
train_cold = train[~train['i'].isin(cold_idx)]
print(f"Films to choose from: {len(eligible):,}")
print(f"Train: {len(train):,} -> {len(train_cold):,} rows after removing the 3,000 films")

cold_mask = torch.zeros(n_films, dtype=torch.bool, device=device)
cold_mask[torch.tensor(cold_idx, dtype=torch.long, device=device)] = True

def relevant_cold(df):
    """Liked films among the new films, per user"""
    liked = df[(df['rating'] >= LIKE) & df['i'].isin(cold_idx)]
    return liked.groupby('u')['i'].apply(set).to_dict()

def evaluate_cold(score_fn, eval_df, chunk=500):
    rel = relevant_cold(eval_df)
    ndcgs, recalls = [], []
    for start in range(0, n_users, chunk):
        end = min(start + chunk, n_users)
        u = torch.arange(start, end, device=device)
        scores = score_fn(u).clone()
        scores[:, ~cold_mask] = float('-inf')          # only the new films are candidates
        top = scores.topk(K, dim=1).indices.cpu().numpy()
        for row, uid in enumerate(range(start, end)):
            if uid not in rel:
                continue
            r = rel[uid]
            hits = [1 if j in r else 0 for j in top[row]]
            dcg = sum(h / np.log2(p + 2) for p, h in enumerate(hits))
            idcg = sum(1 / np.log2(p + 2) for p in range(min(len(r), K)))
            ndcgs.append(dcg / idcg)
            recalls.append(sum(hits) / len(r))
    return {'NDCG@10': round(float(np.mean(ndcgs)), 4),
            'Recall@10': round(float(np.mean(recalls)), 4),
            'Users': len(ndcgs)}

def make_profile(train_df):
    """User profile (mean text vector of liked films) from a given train set"""
    p = train_df[train_df['rating'] >= LIKE]
    pu = torch.tensor(p['u'].values, dtype=torch.long, device=device)
    pi = torch.tensor(p['i'].values, dtype=torch.long, device=device)
    prof = torch.zeros(n_users, film_text.shape[1], device=device)
    prof.index_add_(0, pu, film_text[pi])
    return prof / torch.bincount(pu, minlength=n_users).clamp(min=1).unsqueeze(1), pu, pi

profile_cold, pos_u_c, pos_i_c = make_profile(train_cold)

torch.manual_seed(SEED)
print("Random:           ", evaluate_cold(lambda u: torch.rand(len(u), n_films, device=device), val))
print("Content (no training):", evaluate_cold(lambda u: profile_cold[u] @ film_text.T, val))

Films to choose from: 22,312
Train: 3,552,662 -> 3,124,271 rows after removing the 3,000 films
Random:            {'NDCG@10': 0.0029, 'Recall@10': 0.0029, 'Users': 2853}
Content (no training): {'NDCG@10': 0.0074, 'Recall@10': 0.0082, 'Users': 2853}


### Retraining the two-tower model without the new films

We train a fresh two-tower model on `train_cold` (8 epochs, the number of epochs that worked well before, so we do not need to choose on the new films). The new films are blocked from being drawn as negatives with `seen_mask_c[:, cold_mask] = True`.

The result is the table below. The two-tower model gets NDCG@10 **0.0406**, about 14 times better than random (0.0029) and 5.5 times better than the content-only baseline (0.0074). So the model has learned something useful about films it has never seen a single rating for.

In [9]:
# "Seen" table for train_cold. The new films are never drawn as negatives
seen_mask_c = torch.zeros(n_users, n_films, dtype=torch.bool, device=device)
seen_mask_c[torch.tensor(train_cold['u'].values, dtype=torch.long, device=device),
            torch.tensor(train_cold['i'].values, dtype=torch.long, device=device)] = True
seen_mask_c[:, cold_mask] = True

def sample_neg(u, mask, n_neg=4):
    u = u.repeat_interleave(n_neg)
    neg = torch.randint(0, n_films, u.shape, device=device)
    bad = mask[u, neg]
    while bad.any():
        neg[bad] = torch.randint(0, n_films, (int(bad.sum()),), device=device)
        bad = mask[u, neg]
    return u, neg

def train_two_tower(pos_u, pos_i, profile, mask, epochs=8):
    torch.manual_seed(SEED)
    m = TwoTower(film_text, profile).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(1, epochs + 1):
        m.train()
        u_neg, i_neg = sample_neg(pos_u, mask, N_NEG)
        u_all, i_all = torch.cat([pos_u, u_neg]), torch.cat([pos_i, i_neg])
        y_all = torch.cat([torch.ones(len(pos_u)), torch.zeros(len(u_neg))]).to(device)
        perm = torch.randperm(len(y_all), device=device)
        total = 0.0
        for s in range(0, len(perm), BATCH):
            idx = perm[s:s + BATCH]
            loss = loss_fn(m(u_all[idx], i_all[idx]), y_all[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        print(f"  epoch {epoch}: loss {total / len(perm):.4f}")
    return m

def tt_score_fn(m):
    """Score function for evaluate_cold: user vectors times all film vectors"""
    m.eval()
    with torch.no_grad():
        fv = m.film_vec(torch.arange(n_films, device=device))
    def fn(u):
        with torch.no_grad():
            return m.user_vec(u) @ fv.T
    return fn

tt_cold = train_two_tower(pos_u_c, pos_i_c, profile_cold, seen_mask_c)

torch.manual_seed(SEED)
cold_results = {
    'Random': evaluate_cold(lambda u: torch.rand(len(u), n_films, device=device), val),
    'Content (no training)': evaluate_cold(lambda u: profile_cold[u] @ film_text.T, val),
    'Two-tower (text + ID)': evaluate_cold(tt_score_fn(tt_cold), val),
}
cold_tbl = pd.DataFrame(cold_results).T
print(cold_tbl)
cold_tbl.to_csv(BASE + 'results_cold_val.csv')

  epoch 1: loss 0.3200
  epoch 2: loss 0.2636
  epoch 3: loss 0.2429
  epoch 4: loss 0.2279
  epoch 5: loss 0.2181
  epoch 6: loss 0.2101
  epoch 7: loss 0.2036
  epoch 8: loss 0.1983
                       NDCG@10  Recall@10   Users
Random                  0.0029     0.0029  2853.0
Content (no training)   0.0074     0.0082  2853.0
Two-tower (text + ID)   0.0406     0.0415  2853.0


### Controls: where does the improvement come from?

0.0406 is much better than random, but *why*? Two controls answer that:

1. **Non-personalized:** take the trained model, average all user vectors into one "typical user" and give **everyone the same list**. If this scores almost as well, most of the gain comes from knowing which kinds of films are liked by most people, not from personalization.
2. **ID only (no text):** train the same model, but with the text vectors set to zero. The model then knows films only by their id, and a new film has no id signal. This should be close to random.

Results: the non-personalized list gets 0.0380, so more than 90% of the gain over random comes from **general taste** (for example that well-known franchises and certain genres are liked by most people). Personalization adds only a little: from 0.0380 to 0.0406 (+7%). The ID-only model scores 0.0031, the same as random, which shows that **the text is what makes cold start possible**.

Caveat: in an earlier run with the same settings the two-tower model scored 0.0267 and the non-personalized list 0.0200, so the size of the personalization effect varies between runs and should not be over-interpreted.

In [10]:
# 1. Non-personalized version of the trained model: the same ranking for everyone
tt_cold.eval()
with torch.no_grad():
    fv = tt_cold.film_vec(torch.arange(n_films, device=device))
    mean_u = tt_cold.user_vec(torch.arange(n_users, device=device)).mean(0, keepdim=True)
    same_for_all = mean_u @ fv.T
cold_results['Two-tower, non-personalized (same list for all)'] = evaluate_cold(
    lambda u: same_for_all.repeat(len(u), 1), val)

# 2. ID only, no text (the text vectors are set to zero temporarily)
film_text_real = film_text
film_text = torch.zeros_like(film_text_real)
try:
    tt_id = train_two_tower(pos_u_c, pos_i_c, torch.zeros_like(profile_cold), seen_mask_c)
finally:
    film_text = film_text_real          # always restore the real text vectors
cold_results['Two-tower, ID only (no text)'] = evaluate_cold(tt_score_fn(tt_id), val)

cold_tbl = pd.DataFrame(cold_results).T
print(cold_tbl)
cold_tbl.to_csv(BASE + 'results_cold_val.csv')

  epoch 1: loss 0.3390
  epoch 2: loss 0.3100
  epoch 3: loss 0.3034
  epoch 4: loss 0.2941
  epoch 5: loss 0.2894
  epoch 6: loss 0.2853
  epoch 7: loss 0.2808
  epoch 8: loss 0.2762
                                                 NDCG@10  Recall@10   Users
Random                                            0.0029     0.0029  2853.0
Content (no training)                             0.0074     0.0082  2853.0
Two-tower (text + ID)                             0.0406     0.0415  2853.0
Two-tower, non-personalized (same list for all)   0.0380     0.0399  2853.0
Two-tower, ID only (no text)                      0.0031     0.0034  2853.0


## Final evaluation on the test set

Nothing about the test set has been used for any choice so far. We now evaluate once, in two parts:

1. **The normal task**, with the model chosen on validation (best epoch 10). As before, films seen in train or validation are excluded. The row is added to `results_test.csv`.
2. **The cold start experiment**, evaluated on the *test* ratings of the 3,000 new films, with the same models as above. The profile still comes from the reduced train set, which is the strictest setup. The results go to `results_cold_test.csv`.

Test results:

- **Normal task:** NCF 0.3393 > two-tower 0.3275 > popularity 0.2562 > SVD 0.2377 (NDCG@10). The two-tower model is about 28% better than popularity but slightly below NCF, so the text did not help when all films have many ratings.
- **Cold start:** the two-tower model gets 0.0379, about 12 times random (0.0031) and 3.8 times content-only (0.0100). The non-personalized list gets 0.0369, so personalization adds only about 3%. The ID-only model (0.0023) is no better than random. Conclusion: for new films the text lets the model recommend films that are generally liked, but we cannot show a real personalization effect.

In [11]:
TEST_PATH = BASE + 'results_test.csv'
COLS = ['NDCG@10', 'Recall@10', 'Coverage']

def add_test_row(name, res):
    """Add (or replace) a row in results_test.csv"""
    if os.path.exists(TEST_PATH):
        df = pd.read_csv(TEST_PATH, index_col=0)
    else:
        df = pd.DataFrame(columns=COLS)
    df.loc[name] = [res[c] for c in COLS]
    df.to_csv(TEST_PATH)
    return df

# Films each user has seen in train + validation (excluded at test time)
seen_tv = data[data['split'].isin(['train', 'val'])] \
    .groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

# Normal task: the two-tower model with the best epoch chosen on validation
recs_test = recommend_two_tower(tt, exclude=('train', 'val'))
res = evaluate(lambda user, seen, k: recs_test[user], test, seen_tv)
df_test = add_test_row('Two-tower', res)
df_test.round(4)

,NDCG@10,Recall@10,Coverage
Popularity (likes),0.2562,0.0493,0.0024
SVD (min 0 ratings),0.0106,0.0022,0.0324
SVD (min 1500 ratings),0.2377,0.0459,0.0010
NCF (NeuMF),0.3393,0.0685,0.0387
Two-tower,0.3275,0.0641,0.0300


In [12]:
torch.manual_seed(SEED)
cold_test = {
    'Random': evaluate_cold(lambda u: torch.rand(len(u), n_films, device=device), test),
    'Content (no training)': evaluate_cold(lambda u: profile_cold[u] @ film_text.T, test),
    'Two-tower (text + ID)': evaluate_cold(tt_score_fn(tt_cold), test),
    'Two-tower, non-personalized (same list for all)': evaluate_cold(
        lambda u: same_for_all.repeat(len(u), 1), test),
    'Two-tower, ID only (no text)': evaluate_cold(tt_score_fn(tt_id), test),
}
cold_test_tbl = pd.DataFrame(cold_test).T
cold_test_tbl.to_csv(BASE + 'results_cold_test.csv')
cold_test_tbl

,NDCG@10,Recall@10,Users
Random,0.0031,0.0034,2865.0
Content (no training),0.0100,0.0116,2865.0
Two-tower (text + ID),0.0379,0.0411,2865.0
"Two-tower, non-personalized (same list for all)",0.0369,0.0388,2865.0
"Two-tower, ID only (no text)",0.0023,0.0023,2865.0


## Export for the demo app

The optional Streamlit demo app does not run any model. It only shows **precomputed** recommendations, so we save them as small files in `app_data/`:

- `recs_popularity.json` and `recs_two_tower.json`: the top 10 films per user from the two models. Films seen in train or validation are excluded, as if the recommendations were made "now".
- `history.json`: up to 12 films each user liked, so the app can show a taste profile next to the recommendations.
- `films.csv`: title, year, genres and poster link for each film.

Users are **pseudonymized** ("User 1", "User 2", ...) so no Letterboxd usernames are published.

In [13]:
users = data['user_name'].cat.categories
pseudo = {u: f"User {k + 1}" for k, u in enumerate(users)}

# Recommendations "now": films seen in train and validation are excluded
recs_pop_app = {u: recommend_pop(u, seen_tv.get(u, set()), K) for u in users}
recs_tt_app = recommend_two_tower(tt, exclude=('train', 'val'))

def to_app(recs):
    return {pseudo[u]: [str(f) for f in fs] for u, fs in recs.items()}

# A few liked films per user, to show in the app
liked = data[data['split'].isin(['train', 'val']) & (data['rating'] >= LIKE)]
hist = {pseudo[u]: g['film_id'].astype(str).sample(min(12, len(g)), random_state=SEED).tolist()
        for u, g in liked.groupby('user_name', observed=True)}

films_app = meta[['film_id', 'film_name', 'year', 'genres']].merge(
    films[['film_id', 'poster_url']], on='film_id', how='left')

os.makedirs(BASE + 'app_data', exist_ok=True)
with open(BASE + 'app_data/recs_popularity.json', 'w') as f:
    json.dump(to_app(recs_pop_app), f)
with open(BASE + 'app_data/recs_two_tower.json', 'w') as f:
    json.dump(to_app(recs_tt_app), f)
with open(BASE + 'app_data/history.json', 'w') as f:
    json.dump(hist, f)
films_app.to_csv(BASE + 'app_data/films.csv', index=False)

print(list(hist.items())[0])
print(list(to_app(recs_tt_app).items())[0])
print(f"Files saved in {BASE}app_data/")

('User 1', ['the-first-wives-club', 'lady-bird', 'spider-man-across-the-spider-verse', 'batman-under-the-red-hood', 'maleficent', 'lost-ollie', 'mulan-2020', 'eyes-without-a-face', 'my-neighbor-totoro', 'encanto', 'the-many-adventures-of-winnie-the-pooh', 'you-again'])
('User 1', ['frozen-2013', 'monsters-inc', 'barbie', 'shrek', 'home-alone', 'la-la-land', 'elf', '500-days-of-summer', 'interstellar', 'eternal-sunshine-of-the-spotless-mind'])
Files saved in /content/drive/MyDrive/recsys/app_data/
